In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
random.seed(SEED)
np.random.seed(SEED)
packages = ["numpy", "pandas", "scikit-learn", "lightgbm", "catboost", "pyarrow", "matplotlib", "joblib"]
environment = {"python": platform.python_version(), "platform": platform.platform(), "mode": MODE}
for package in packages:
    try:
        environment[package] = version(package)
    except PackageNotFoundError:
        environment[package] = "not installed"
if FULL_RUN:
    import joblib
    from lightgbm import LGBMClassifier, early_stopping, log_evaluation
    from catboost import CatBoostClassifier
    import pyarrow
print(json.dumps(environment, indent=2))
(OUTPUT_DIR / "environment.json").write_text(json.dumps(environment, indent=2))


ModuleNotFoundError: No module named 'numpy'

In [ ]:
from pathlib import Path
import sys, json, hashlib, platform, time, random
from importlib.metadata import version, PackageNotFoundError

MODE = "review"  
PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "notebook_output"
INPUTS = {
    "train_signals": DATA_DIR / "train_signals.csv",
    "train_transactions": DATA_DIR / "train_transactions.parquet",
    "test_signals": DATA_DIR / "test_signals.csv",
    "test_transactions": DATA_DIR / "test_transactions.parquet",
    "sample": DATA_DIR / "sample_submission.csv",
}
assert MODE in {"review", "full"}
FULL_RUN = MODE == "full"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR = OUTPUT_DIR if FULL_RUN else PROJECT_DIR / "artifacts"
SEED = 42
WINDOWS = (1, 3, 7, 14, 30, 60, 90, 180)
DIRECTIONS = ("kirim", "chiqim")
TYPES = ("karta", "bank_otkazmasi", "naqd", "xalqaro")
print("Mode:", MODE, "| Project:", PROJECT_DIR)


Mode: review | Project: /Users/dianafatykhova/Downloads/escalation_project


In [ ]:
def read_table(path):
    path = Path(path)
    return pd.read_parquet(path) if path.suffix.lower() == '.parquet' else pd.read_csv(path)

def load_signals(path, training=False):
    d = read_table(path)
    required = ['signal_id', 'signal_sanasi'] + (['eskalatsiya'] if training else [])
    if not set(required).issubset(d):
        raise ValueError(f'Missing signal columns: {set(required) - set(d)}')
    if d[required].isna().any().any() or d.signal_id.duplicated().any():
        raise ValueError('Signal identifiers, dates and targets must be present; identifiers must be unique.')
    d['signal_sanasi'] = pd.to_datetime(d.signal_sanasi, errors='raise')
    if training and (not set(d.eskalatsiya.unique()).issubset({0, 1})):
        raise ValueError('Target must be binary.')
    return d

def load_transactions(path):
    cols = ['signal_id', 'tranzaksiya_vaqti', 'kirim_chiqim', 'tranzaksiya_turi', 'miqdor_indeksi']
    if Path(path).suffix.lower() == '.parquet':
        d = pd.read_parquet(path, columns=cols)
    else:
        d = pd.read_csv(path, usecols=cols, dtype={'kirim_chiqim': 'category', 'tranzaksiya_turi': 'category', 'miqdor_indeksi': 'float64'})
    return d


In [ ]:
if FULL_RUN:
    missing = [str(p) for p in INPUTS.values() if not p.is_file()]
    if missing:
        raise FileNotFoundError("Add the five organizer files or correct INPUTS:\n" + "\n".join(missing))
    train_signals = load_signals(INPUTS["train_signals"], training=True)
    test_signals = load_signals(INPUTS["test_signals"])
    sample = pd.read_csv(INPUTS["sample"])
    assert set(train_signals.eskalatsiya.unique()) == {0, 1}, "Training needs both classes."
    assert not set(train_signals.signal_id) & set(test_signals.signal_id)
    assert sample.columns.tolist() in (["signal_id", "eskalatsiya"], ["signal_id", "ehtimollik"]), "Expected a two-column probability template."
    sample = sample.rename(columns={"ehtimollik": "eskalatsiya"})
    manifest = {}
    for name, path in INPUTS.items():
        h = hashlib.sha256()
        with path.open("rb") as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                h.update(chunk)
        manifest[name] = {"filename": path.name, "bytes": path.stat().st_size, "sha256": h.hexdigest()}
    (OUTPUT_DIR / "input_manifest.json").write_text(json.dumps(manifest, indent=2))
    print("Training alerts:", len(train_signals), "Test alerts:", len(test_signals))
    print("Escalation rate:", train_signals.eskalatsiya.mean())
else:
    required = [ARTIFACT_DIR / n for n in ["metrics.json", "holdout_predictions.csv", "development_oof.csv"]]
    required.append(PROJECT_DIR / "submission.csv")
    missing = [str(p) for p in required if not p.is_file()]
    if missing:
        raise FileNotFoundError("Extract the supplied project and set PROJECT_DIR:\n" + "\n".join(missing))
    print("Reviewing saved outputs. Original input data were not included in the ZIP.")


In [ ]:
def prepare_transactions(signals, transactions):
    t = transactions.copy()
    raw_count = len(t)
    required = ['signal_id', 'tranzaksiya_vaqti', 'kirim_chiqim', 'tranzaksiya_turi', 'miqdor_indeksi']
    if t[required].isna().any().any():
        raise ValueError('Missing transaction fields; resolve before feature generation.')
    if not set(t.kirim_chiqim.unique()).issubset(DIRECTIONS) or not set(t.tranzaksiya_turi.unique()).issubset(TYPES):
        raise ValueError('Unknown transaction category.')
    if not np.isfinite(t.miqdor_indeksi).all():
        raise ValueError('Non-finite size index.')
    t['tranzaksiya_vaqti'] = pd.to_datetime(t.tranzaksiya_vaqti, errors='raise')
    date_map = signals.set_index('signal_id').signal_sanasi
    dates = t.signal_id.map(date_map)
    if dates.isna().any():
        raise ValueError('Transactions contain unknown signal_id values.')
    t['days_before_signal'] = (dates - t.tranzaksiya_vaqti).dt.total_seconds() / 86400
    audit = {'raw_transactions': raw_count, 'future_transactions_excluded': int((t.days_before_signal < 0).sum()), 'exact_duplicate_rows': int(t[required].duplicated().sum()), 'same_calendar_day_after_midnight_excluded': int(((t.days_before_signal < 0) & (t.days_before_signal > -1)).sum())}
    t = t.loc[t.days_before_signal >= 0].copy()
    t['day'] = t.tranzaksiya_vaqti.dt.floor('D')
    t['night'] = (t.tranzaksiya_vaqti.dt.hour < 6).astype('int8')
    t['weekend'] = (t.tranzaksiya_vaqti.dt.dayofweek >= 5).astype('int8')
    t['large'] = (t.miqdor_indeksi > 2).astype('int8')
    audit.update({'usable_transactions': len(t), 'alerts_without_history': int((~signals.signal_id.isin(t.signal_id)).sum()), 'min_size_index': float(t.miqdor_indeksi.min()), 'max_size_index': float(t.miqdor_indeksi.max()), 'negative_size_fraction': float((t.miqdor_indeksi < 0).mean())})
    return (t, audit)

def build_features(signals, transactions):
    t, audit = prepare_transactions(signals, transactions)
    idx = pd.Index(signals.signal_id, name='signal_id')
    pieces = []

    def add(frame, prefix):
        pieces.append(frame.reindex(idx).add_prefix(prefix))
    for window in (*WINDOWS, None):
        name = 'all' if window is None else f'{window}d'
        q = t if window is None else t[t.days_before_signal <= window]
        g = q.groupby('signal_id', observed=True)
        basic = g.agg(count=('miqdor_indeksi', 'size'), size_mean=('miqdor_indeksi', 'mean'), size_std=('miqdor_indeksi', 'std'), size_min=('miqdor_indeksi', 'min'), size_max=('miqdor_indeksi', 'max'), size_skew=('miqdor_indeksi', 'skew'), active_days=('day', 'nunique'), night_share=('night', 'mean'), weekend_share=('weekend', 'mean'), large_share=('large', 'mean'))
        quant = g.miqdor_indeksi.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).unstack().reindex(columns=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
        quant.columns = ['size_q10', 'size_q25', 'size_median', 'size_q75', 'size_q90', 'size_q99']
        basic = basic.join(quant)
        basic['size_iqr'] = basic.size_q75 - basic.size_q25
        basic['transactions_per_active_day'] = basic['count'] / basic.active_days
        add(basic, name + '_')
        for col, cats in [('kirim_chiqim', DIRECTIONS), ('tranzaksiya_turi', TYPES)]:
            for value in cats:
                h = q[q[col] == value].groupby('signal_id', observed=True).miqdor_indeksi
                z = h.agg(['count', 'mean', 'std', 'max'])
                z['share'] = z['count'] / basic['count']
                add(z, f'{name}_{value}_')
        if window in (7, 30, 90, None):
            for direction in DIRECTIONS:
                for typ in TYPES:
                    h = q[(q.kirim_chiqim == direction) & (q.tranzaksiya_turi == typ)].groupby('signal_id', observed=True).miqdor_indeksi
                    add(h.agg(['count', 'mean']), f'{name}_{direction}_{typ}_')
            daily = q.groupby(['signal_id', 'day'], observed=True).size().groupby(level=0).agg(['max', 'mean', 'std'])
            add(daily, name + '_daily_count_')
    g = t.groupby('signal_id', observed=True)
    add(g.days_before_signal.agg(['min', 'max', 'mean', 'std']), 'recency_')
    ordered = t.sort_values(['signal_id', 'tranzaksiya_vaqti'])
    ordered['gap_hours'] = ordered.groupby('signal_id', observed=True).tranzaksiya_vaqti.diff().dt.total_seconds() / 3600
    add(ordered.groupby('signal_id', observed=True).gap_hours.agg(['min', 'max', 'mean', 'std', 'median']), 'gap_hours_')
    last = ordered.groupby('signal_id', observed=True).tail(1).set_index('signal_id')
    add(last[['miqdor_indeksi']].rename(columns={'miqdor_indeksi': 'size'}), 'last_')
    for typ in TYPES:
        rec = t[t.tranzaksiya_turi == typ].groupby('signal_id', observed=True).days_before_signal.min()
        add(rec.to_frame('days'), f'last_{typ}_')
    X = pd.concat(pieces, axis=1)
    zero_columns = [c for c in X if c.endswith(('_count', '_share', '_active_days')) or '_count_' in c]
    X[zero_columns] = X[zero_columns].fillna(0)
    extra = {}
    for short, long in [(1, 7), (3, 30), (7, 30), (7, 90), (14, 90), (30, 90), (30, 180)]:
        a, b = (X[f'{short}d_count'], X[f'{long}d_count'])
        extra[f'activity_rate_ratio_{short}_{long}'] = (a + 0.5) / short / ((b - a + 0.5) / (long - short))
        extra[f'activity_share_{short}_{long}'] = a / (b + 1)
        extra[f'size_mean_change_{short}_{long}'] = X[f'{short}d_size_mean'] - X[f'{long}d_size_mean']
        for typ in TYPES:
            extra[f'{typ}_share_change_{short}_{long}'] = X[f'{short}d_{typ}_share'] - X[f'{long}d_{typ}_share']
    for w in (*WINDOWS, None):
        n = 'all' if w is None else f'{w}d'
        extra[n + '_direction_imbalance'] = (X[n + '_kirim_count'] - X[n + '_chiqim_count']) / (X[n + '_count'] + 1)
        proportions = X[[f'{n}_{v}_share' for v in TYPES]].to_numpy()
        extra[n + '_type_entropy'] = -(proportions * np.log(np.clip(proportions, 1e-12, 1))).sum(axis=1)
    extra['history_observed_days'] = X.recency_max - X.recency_min
    X = pd.concat([X, pd.DataFrame(extra, index=idx)], axis=1).astype('float32')
    X = X.replace([np.inf, -np.inf], np.nan)
    audit['feature_count'] = X.shape[1]
    return (X, audit)


In [ ]:
check_signals = pd.DataFrame({"signal_id": ["a", "b"], "signal_sanasi": pd.to_datetime(["2026-01-10", "2026-01-10"])})
check_transactions = pd.DataFrame({
    "signal_id": ["a", "a", "a"],
    "tranzaksiya_vaqti": ["2026-01-09", "2026-01-10 00:00:00", "2026-01-10 12:00:00"],
    "kirim_chiqim": ["kirim"] * 3,
    "tranzaksiya_turi": ["karta"] * 3,
    "miqdor_indeksi": [1., 2., 999.],
})
# Convert mixed timestamp strings explicitly before the project loader processes them.
check_transactions["tranzaksiya_vaqti"] = pd.to_datetime(check_transactions.tranzaksiya_vaqti, format="mixed")
check_X, check_audit = build_features(check_signals, check_transactions)
without_future, _ = build_features(check_signals, check_transactions.iloc[:2])
pd.testing.assert_frame_equal(check_X, without_future)
assert check_audit["future_transactions_excluded"] == 1
assert check_X.loc["a", "1d_count"] == 2
assert check_X.loc["b", "all_count"] == 0
assert not {"eskalatsiya", "signal_id", "signal_sanasi"} & set(check_X.columns)
print("PASS: future exclusion, inclusive cutoff, missing history, and target exclusion.")
print("Feature columns:", check_X.shape[1])


In [ ]:
if FULL_RUN:
    for split, signals in [("train", train_signals), ("test", test_signals)]:
        transactions = load_transactions(INPUTS[split + "_transactions"])
        features, audit = build_features(signals, transactions)
        assert features.index.tolist() == signals.signal_id.tolist()
        assert features.columns.is_unique
        features.to_parquet(OUTPUT_DIR / f"{split}_features.parquet")
        (OUTPUT_DIR / f"{split}_features.audit.json").write_text(json.dumps(audit, indent=2))
        print(split, features.shape, audit)
        del transactions, features
    X = pd.read_parquet(OUTPUT_DIR / "train_features.parquet")
    X_test = pd.read_parquet(OUTPUT_DIR / "test_features.parquet")
    assert X.columns.tolist() == X_test.columns.tolist()
else:
    print("Feature generation skipped in REVIEW mode; the small code check above was run.")


In [ ]:
def metrics(y, p):
    return {'roc_auc': float(roc_auc_score(y, p)), 'average_precision': float(average_precision_score(y, p)), 'log_loss': float(log_loss(y, p, labels=[0, 1])), 'brier_score': float(brier_score_loss(y, p))}

def bootstrap_auc(y, p, n=1000):
    y, p = (np.asarray(y), np.asarray(p))
    rng = np.random.default_rng(SEED)
    scores = []
    for _ in range(n):
        ix = rng.integers(0, len(y), len(y))
        if len(np.unique(y[ix])) == 2:
            scores.append(roc_auc_score(y[ix], p[ix]))
    return [float(x) for x in np.quantile(scores, [0.025, 0.975])]

def split_holdout(y):
    return train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=SEED)


In [ ]:
def make_model(name, seed=SEED, rounds=None):
    if name == 'logistic':
        return make_pipeline(SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True), StandardScaler(), LogisticRegression(C=0.05, max_iter=3000, random_state=seed))
    if name == 'lightgbm':
        return LGBMClassifier(n_estimators=rounds or 1600, learning_rate=0.025, num_leaves=15, max_depth=-1, min_child_samples=70, colsample_bytree=0.85, subsample=0.85, subsample_freq=1, reg_lambda=12, reg_alpha=0.1, random_state=seed, n_jobs=4, verbosity=-1, deterministic=True, force_col_wise=True)
    if name == 'catboost':
        return CatBoostClassifier(iterations=rounds or 1600, depth=5, learning_rate=0.035, l2_leaf_reg=10, loss_function='Logloss', eval_metric='AUC', random_seed=seed, thread_count=4, verbose=False, allow_writing_files=False)
    raise ValueError(name)

def fit_model(name, X, y, seed=SEED, rounds=None, validation=None):
    m = make_model(name, seed, rounds)
    if validation is not None and name == 'lightgbm':
        m.fit(X, y, eval_set=[validation], eval_metric='auc', callbacks=[early_stopping(100, verbose=False), log_evaluation(0)])
        r = m.best_iteration_
    elif validation is not None and name == 'catboost':
        m.fit(X, y, eval_set=validation, early_stopping_rounds=100)
        r = m.tree_count_
    else:
        m.fit(X, y)
        r = rounds
    return (m, r)

def predict_model(m, X):
    return m.predict_proba(X)[:, 1]


In [ ]:
def train_pipeline(X, signals, out):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    s = signals.set_index('signal_id').loc[X.index]
    y = s.eskalatsiya.to_numpy()
    dev, hold = split_holdout(y)
    report = {'seed': SEED, 'features': X.shape[1], 'development_rows': len(dev), 'holdout_rows': len(hold), 'validation_protocol': '3-fold stratified development CV; untouched 20% random holdout; separate temporal stress test', 'candidates': {}, 'rounds': {}}
    predictions = {}
    rounds = {}
    names = ['logistic', 'lightgbm', 'catboost']
    folds = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(X.iloc[dev], y[dev]))
    for name in names:
        p = np.zeros(len(dev))
        rs = []
        fold_metrics = []
        for f, (tr, va) in enumerate(folds):
            started = time.time()
            m, r = fit_model(name, X.iloc[dev[tr]], y[dev[tr]], SEED + f, validation=(X.iloc[dev[va]], y[dev[va]]))
            p[va] = predict_model(m, X.iloc[dev[va]])
            rs.append(r)
            fold_metrics.append(metrics(y[dev[va]], p[va]))
            print(name, 'fold', f + 1, 'rounds', r, 'AUC', fold_metrics[-1]['roc_auc'], 'seconds', round(time.time() - started, 1), flush=True)
        rounds[name] = int(np.median(rs)) if name != 'logistic' else None
        predictions[name] = p
        report['candidates'][name] = {'pooled_oof': metrics(y[dev], p), 'folds': fold_metrics}
    predictions['blend'] = (predictions['lightgbm'] + predictions['catboost']) / 2
    report['candidates']['blend'] = {'pooled_oof': metrics(y[dev], predictions['blend'])}
    selected = max(predictions, key=lambda n: report['candidates'][n]['pooled_oof']['roc_auc'])
    single = max(names, key=lambda n: report['candidates'][n]['pooled_oof']['roc_auc'])
    if selected == 'blend' and report['candidates']['blend']['pooled_oof']['roc_auc'] - report['candidates'][single]['pooled_oof']['roc_auc'] < 0.001:
        selected = single
    parts = ['lightgbm', 'catboost'] if selected == 'blend' else [selected]
    report['selected'] = selected
    report['rounds'] = rounds
    print('Selected:', selected, rounds, flush=True)
    holdp = np.zeros(len(hold))
    importance = []
    for name in parts:
        m, _ = fit_model(name, X.iloc[dev], y[dev], rounds=rounds[name])
        holdp += predict_model(m, X.iloc[hold]) / len(parts)
        if hasattr(m, 'feature_importances_'):
            importance.append(pd.Series(m.feature_importances_, index=X.columns, name=name))
    report['holdout'] = metrics(y[hold], holdp)
    report['holdout']['roc_auc_95pct_bootstrap'] = bootstrap_auc(y[hold], holdp)
    report['holdout_constant_baseline'] = metrics(y[hold], np.full(len(hold), y[dev].mean()))
    pd.DataFrame({'signal_id': X.index[hold], 'target': y[hold], 'probability': holdp}).to_csv(out / 'holdout_predictions.csv', index=False)
    pd.DataFrame({'signal_id': X.index[dev], 'target': y[dev], **predictions}).to_csv(out / 'development_oof.csv', index=False)
    if importance:
        z = pd.concat(importance, axis=1)
        z = z.div(z.sum(axis=0), axis=1)
        z['mean'] = z.mean(axis=1)
        z.sort_values('mean', ascending=False).to_csv(out / 'feature_importance.csv')
    cutoff = s.signal_sanasi.quantile(0.8)
    tr = np.flatnonzero(s.signal_sanasi < cutoff)
    va = np.flatnonzero(s.signal_sanasi >= cutoff)
    tp = np.zeros(len(va))
    for name in parts:
        m, _ = fit_model(name, X.iloc[tr], y[tr], rounds=rounds[name])
        tp += predict_model(m, X.iloc[va]) / len(parts)
    report['temporal_stress_test'] = {**metrics(y[va], tp), 'cutoff': str(cutoff), 'train_rows': len(tr), 'validation_rows': len(va), 'caveat': 'Sensitivity check only: model selection used the random development split, overlapping this period.'}
    print('Holdout:', report['holdout'], 'Temporal:', report['temporal_stress_test'], flush=True)
    models = []
    for seed in [42, 137, 2026]:
        for name in parts:
            m, _ = fit_model(name, X, y, seed=seed, rounds=rounds[name])
            models.append(m)
    bundle = {'models': models, 'columns': list(X.columns), 'selected': selected, 'rounds': rounds, 'feature_schema_version': 1}
    joblib.dump(bundle, out / 'model.joblib', compress=3)
    report['final_model_count'] = len(models)
    report['final_training_rows'] = len(X)
    report['probability_note'] = 'Unweighted Logloss training; no post-hoc calibration. Brier/log loss and reliability plot assess calibration. ROC-AUC measures ranking, not probability correctness.'
    report['limitations'] = ['No customer identifier: cannot enforce customer-disjoint validation.', 'Date-only alert timestamps use midnight, excluding same-day later transactions.', 'Development CV with early stopping is used for selection; untouched holdout is the primary estimate.', 'Final all-data seed ensemble is not independently evaluated; reported holdout evaluates the selected recipe.', 'Hidden-test ROC-AUC is unknown; synthetic findings are not evidence of real-world suspicious behavior.']
    (out / 'metrics.json').write_text(json.dumps(report, indent=2))
    print('Saved model and metrics to', out, flush=True)
    return (report, bundle)


In [ ]:
if FULL_RUN:
    report, bundle = train_pipeline(X, train_signals, OUTPUT_DIR)
    print("Fresh training finished.")
else:
    report = json.loads((ARTIFACT_DIR / "metrics.json").read_text())
    print("Loaded the original training report; no model was retrained in REVIEW mode.")


In [ ]:
holdout = pd.read_csv(ARTIFACT_DIR / "holdout_predictions.csv")
oof = pd.read_csv(ARTIFACT_DIR / "development_oof.csv")
assert holdout.signal_id.is_unique and oof.signal_id.is_unique
assert not set(holdout.signal_id) & set(oof.signal_id)
recalculated = metrics(holdout.target, holdout.probability)
for key, value in recalculated.items():
    assert np.isclose(value, report["holdout"][key], atol=1e-10, rtol=0), key
comparison = []
for name, details in report["candidates"].items():
    actual = metrics(oof.target, oof[name])
    for key, value in actual.items():
        assert np.isclose(value, details["pooled_oof"][key], atol=1e-10, rtol=0), (name, key)
    comparison.append({"model": name, **actual})
comparison = pd.DataFrame(comparison).set_index("model")
print(comparison.round(6).to_string())
print("Selected:", report["selected"])
print("Recalculated holdout:", json.dumps(recalculated, indent=2))
print("Stored holdout 95% bootstrap interval:", report["holdout"]["roc_auc_95pct_bootstrap"])
print("Temporal sensitivity ROC-AUC (stored):", report["temporal_stress_test"]["roc_auc"])
print("PASS: all reported holdout and development metrics match prediction CSVs.")


In [ ]:
from sklearn.metrics import roc_curve
from sklearn.calibration import calibration_curve
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
comparison.roc_auc.plot.barh(ax=axes[0], color="#267a8a")
axes[0].axvline(0.5, color="gray", linestyle="--")
axes[0].set(title="Development model comparison", xlabel="ROC-AUC", xlim=(0.45, 0.75))
fpr, tpr, _ = roc_curve(holdout.target, holdout.probability)
axes[1].plot(fpr, tpr, label=f"AUC = {recalculated['roc_auc']:.4f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set(title="Holdout ranking", xlabel="False positive rate", ylabel="True positive rate")
axes[1].legend()
observed, predicted = calibration_curve(holdout.target, holdout.probability, n_bins=8, strategy="quantile")
axes[2].plot(predicted, observed, "o-")
axes[2].plot([0, 1], [0, 1], "--", color="gray")
axes[2].set(title="Holdout probability calibration", xlabel="Mean predicted probability", ylabel="Observed escalation fraction")
fig.suptitle("Fresh run" if FULL_RUN else "Recomputed from supplied saved predictions")
fig.tight_layout()
plt.show()


In [ ]:
def make_submission(bundle, X, signals, sample):
    if X.index.duplicated().any() or sample.signal_id.duplicated().any():
        raise ValueError('Duplicate identifiers.')
    if set(X.index) != set(signals.signal_id) or set(sample.signal_id) != set(signals.signal_id):
        raise ValueError('Test identifier mismatch.')
    if len(sample.columns) != 2 or sample.columns[0] != 'signal_id':
        raise ValueError('Expected two-column organizer template.')
    if set(X.columns) != set(bundle['columns']):
        raise ValueError('Feature schema mismatch.')
    X = X.loc[sample.signal_id, bundle['columns']]
    p = np.mean([m.predict_proba(X)[:, 1] for m in bundle['models']], axis=0)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError('Invalid probabilities.')
    result = sample.copy()
    result[result.columns[1]] = p
    assert len(result) == len(signals) and result.signal_id.tolist() == sample.signal_id.tolist()
    return result


In [ ]:
if FULL_RUN:
    submission = make_submission(bundle, X_test, test_signals, sample)
    submission_path = OUTPUT_DIR / "submission.csv"
    submission.to_csv(submission_path, index=False, float_format="%.10f")
else:
    original_submission = pd.read_csv(PROJECT_DIR / "submission.csv")
    assert original_submission.columns.tolist() in (["signal_id", "ehtimollik"], ["signal_id", "eskalatsiya"])
    submission = original_submission.rename(columns={"ehtimollik": "eskalatsiya"})
    pd.testing.assert_series_equal(submission.iloc[:, 1], original_submission.iloc[:, 1], check_names=False)
    assert submission.signal_id.tolist() == original_submission.signal_id.tolist()
    submission_path = OUTPUT_DIR / "submission.csv"
    submission.to_csv(submission_path, index=False, float_format="%.10f")
assert submission.columns.tolist() == ["signal_id", "eskalatsiya"]
assert submission.signal_id.notna().all() and submission.signal_id.is_unique
assert len(submission) > 0
assert np.isfinite(submission.eskalatsiya).all()
assert submission.eskalatsiya.between(0, 1).all()
print(submission.head().to_string(index=False))
print("Rows:", len(submission), "Probability range:", submission.eskalatsiya.min(), submission.eskalatsiya.max())
print("Submission:", submission_path)


In [ ]:
verification = {
    "mode": MODE,
    "feature_boundary_check": "passed",
    "prediction_metrics_recalculated": "passed",
    "submission_schema_and_range": "passed",
    "raw_data_pipeline_executed": FULL_RUN,
    "official_test_order_verified": FULL_RUN,
    "holdout_roc_auc": recalculated["roc_auc"],
    "submission_rows": len(submission),
}
(OUTPUT_DIR / "notebook_verification.json").write_text(json.dumps(verification, indent=2))
print(json.dumps(verification, indent=2))
